# Task 4: Churn Prediction Model
### SaiKet Systems — Data Science Internship
**Intern:** Omokhoa Oshose Tosayoname | **ID:** SKS/A2/C115874  
**Project:** Customer Churn Analysis and Prediction  
**Date:** April 2026

---

## Objective
Train and evaluate multiple machine learning classifiers on the Telco Customer Churn dataset.
Apply SMOTE for class imbalance, feature selection via Random Forest importance, and
GridSearchCV hyperparameter tuning. Evaluation metrics: Accuracy, Precision, Recall, F1-Score.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.feature_selection import SelectFromModel

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'axes.titleweight': 'bold'})
CLR_RETAIN = '#2196F3'; CLR_CHURN = '#F44336'; PALETTE = [CLR_RETAIN, CLR_CHURN]
print("Libraries loaded.")


## 2. Load Train / Test Splits

In [ ]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f"Training set   : {X_train.shape[0]:,} samples × {X_train.shape[1]} features")
print(f"Testing set    : {X_test.shape[0]:,} samples × {X_test.shape[1]} features")
print(f"Churn rate — train : {y_train.mean()*100:.2f}%")
print(f"Churn rate — test  : {y_test.mean()*100:.2f}%")


## 3. Handling Class Imbalance with SMOTE
The dataset has a ~26.5% churn rate. Without correction, classifiers bias toward the
majority class, producing misleadingly high accuracy but poor recall on churned customers
— the class we care most about. **SMOTE** generates synthetic minority samples on the
training set only, balancing classes to 50/50.


In [ ]:
print("Class distribution BEFORE SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nClass distribution AFTER SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print(f"New training size: {len(X_train_bal):,} (balanced 50/50)")


## 4. Feature Scaling
**StandardScaler** is fitted on the SMOTE-balanced training set, then applied to
both train and test to prevent data leakage.


In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc  = scaler.transform(X_test)
print(f"X_train_sc : {X_train_sc.shape}")
print(f"X_test_sc  : {X_test_sc.shape}")


## 5. Baseline Model Comparison
Six classifiers are benchmarked with default parameters using 5-fold stratified CV
and held-out test evaluation.


In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree'       : DecisionTreeClassifier(random_state=SEED),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    'SVM'                 : SVC(probability=True, random_state=SEED),
    'KNN'                 : KNeighborsClassifier()
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

print(f"{'Model':<22} {'CV F1':>7} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("-" * 58)

for name, model in models.items():
    cv_f1 = cross_val_score(model, X_train_sc, y_train_bal,
                            cv=cv, scoring='f1').mean()
    model.fit(X_train_sc, y_train_bal)
    y_pred = model.predict(X_test_sc)
    r = {'Model': name, 'CV_F1': cv_f1,
         'Accuracy': accuracy_score(y_test, y_pred),
         'Precision': precision_score(y_test, y_pred),
         'Recall': recall_score(y_test, y_pred),
         'F1': f1_score(y_test, y_pred)}
    results.append(r)
    print(f"{name:<22} {cv_f1:>7.4f} {r['Accuracy']:>7.4f} "
          f"{r['Precision']:>7.4f} {r['Recall']:>7.4f} {r['F1']:>7.4f}")

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(results_df)); width = 0.2
for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, results_df[metric], width,
                label=metric, edgecolor='white', linewidth=0.6)
axes[0].set_xticks(x + width*1.5)
axes[0].set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Score'); axes[0].set_title('Baseline Model Comparison')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, 1.05)
colors_bar = ['#4CAF50' if i==0 else '#90CAF9' for i in range(len(results_df))]
axes[1].barh(results_df['Model'], results_df['F1'], color=colors_bar, edgecolor='white')
axes[1].set_title('Model Ranking by F1-Score'); axes[1].set_xlabel('F1-Score')
for i, (_, row) in enumerate(results_df.iterrows()):
    axes[1].text(row['F1']+0.002, i, f"{row['F1']:.4f}",
                 va='center', fontsize=9, fontweight='bold')
axes[1].set_xlim(0, results_df['F1'].max()*1.15); axes[1].invert_yaxis()
plt.suptitle('Baseline Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task4_01_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Top baseline: {results_df.iloc[0]['Model']}  (F1 = {results_df.iloc[0]['F1']:.4f})")


## 6. Feature Selection
A Random Forest is used to rank all 30 features by importance. Only features above
the mean importance threshold are retained, reducing noise and search space.


In [ ]:
rf_selector = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_selector.fit(X_train_sc, y_train_bal)

importances = pd.Series(rf_selector.feature_importances_,
                        index=X_train.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
top20 = importances.head(20)
colors_imp = ['#F44336' if i < 5 else '#90CAF9' for i in range(len(top20))]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors_imp[::-1], edgecolor='white')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('task4_02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Top 10 features:")
print(importances.head(10).round(4).to_string())


In [ ]:
selector = SelectFromModel(rf_selector, threshold='mean', prefit=True)
X_train_sel = selector.transform(X_train_sc)
X_test_sel  = selector.transform(X_test_sc)
selected_features = X_train.columns[selector.get_support()].tolist()
print(f"Features selected: {len(selected_features)} / {X_train.shape[1]}")
print(f"Selected: {selected_features}")


## 7. Hyperparameter Tuning (GridSearchCV)
The top 3 models are tuned via GridSearchCV with 5-fold stratified CV, optimising
for F1-score. Pre-fitted models are loaded from saved artifacts to avoid rerunning
the full grid search (which takes ~10 minutes on this hardware).

> **Note for reproducibility:** To rerun the full GridSearchCV, uncomment the grid
> search blocks below and comment out the `joblib.load` lines.


In [ ]:
# ─────────────────────────────────────────────────────────────
# To rerun full GridSearchCV, uncomment the blocks below:
# ─────────────────────────────────────────────────────────────
# lr_grid = GridSearchCV(
#     LogisticRegression(max_iter=1000, random_state=SEED),
#     {'C': [0.01, 0.1, 1, 10, 100],
#      'solver': ['lbfgs', 'liblinear'], 'penalty': ['l2']},
#     cv=cv, scoring='f1', n_jobs=-1)
# lr_grid.fit(X_train_sel, y_train_bal)
#
# rf_grid = GridSearchCV(
#     RandomForestClassifier(random_state=SEED),
#     {'n_estimators': [100, 200], 'max_depth': [None, 10, 20],
#      'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]},
#     cv=cv, scoring='f1', n_jobs=-1)
# rf_grid.fit(X_train_sel, y_train_bal)
#
# gb_grid = GridSearchCV(
#     GradientBoostingClassifier(random_state=SEED),
#     {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1, 0.2],
#      'max_depth': [3, 5], 'subsample': [0.8, 1.0]},
#     cv=cv, scoring='f1', n_jobs=-1)
# gb_grid.fit(X_train_sel, y_train_bal)
# ─────────────────────────────────────────────────────────────

# Load pre-fitted best estimators
stage2 = joblib.load('task4_stage2.pkl')

print("GridSearchCV Results:")
print(f"  Logistic Regression — Best params: {stage2['lr_params']}  CV F1: {stage2['lr_cv']:.4f}")
print(f"  Random Forest       — Best params: {stage2['rf_params']}  CV F1: {stage2['rf_cv']:.4f}")
print(f"  Gradient Boosting   — Best params: {stage2['gb_params']}  CV F1: {stage2['gb_cv']:.4f}")


## 8. Tuned Model Evaluation

In [ ]:
tuned_models = stage2['tuned_models']

tuned_results = []
print(f"{'Model':<32} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("-" * 60)
for name, model in tuned_models.items():
    y_pred = model.predict(X_test_sel)
    r = {'Model': name,
         'Accuracy': accuracy_score(y_test, y_pred),
         'Precision': precision_score(y_test, y_pred),
         'Recall': recall_score(y_test, y_pred),
         'F1-Score': f1_score(y_test, y_pred)}
    tuned_results.append(r)
    print(f"{name:<32} {r['Accuracy']:>7.4f} {r['Precision']:>7.4f} "
          f"{r['Recall']:>7.4f} {r['F1-Score']:>7.4f}")

tuned_df = pd.DataFrame(tuned_results).sort_values('F1-Score', ascending=False)
best_model_name = tuned_df.iloc[0]['Model']
best_model = tuned_models[best_model_name]
print(f"\n>>> Best tuned model: {best_model_name}")


## 9. Confusion Matrices — Tuned Models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model) in zip(axes, tuned_models.items()):
    y_pred = model.predict(X_test_sel)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace(' (Tuned)', '\n(Tuned)'), fontsize=10, fontweight='bold')
plt.suptitle('Confusion Matrices — Tuned Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task4_03_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Classification Report — Best Model

In [ ]:
y_pred_best = best_model.predict(X_test_sel)
print(f"=== Classification Report: {best_model_name} ===\n")
print(classification_report(y_test, y_pred_best,
                             target_names=['Retained (0)', 'Churned (1)']))


## 11. Final Model Ranking — Baseline vs Tuned

In [ ]:
all_results = pd.concat([
    results_df.rename(columns={'F1': 'F1-Score'})[
        ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']],
    tuned_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
], ignore_index=True).sort_values('F1-Score', ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))
colors_final = []
for m in all_results['Model']:
    if 'Tuned' in m and m == best_model_name: colors_final.append('#4CAF50')
    elif 'Tuned' in m: colors_final.append('#8BC34A')
    else: colors_final.append('#90CAF9')
bars = ax.barh(all_results['Model'], all_results['F1-Score'],
               color=colors_final, edgecolor='white')
for bar, val in zip(bars, all_results['F1-Score']):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, all_results['F1-Score'].max()*1.15)
ax.set_xlabel('F1-Score', fontsize=11)
ax.set_title('All Models — F1-Score Ranking\n(Green = Tuned | Blue = Baseline)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('task4_04_final_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print(all_results[['Model','F1-Score']].sort_values('F1-Score', ascending=False).to_string(index=False))


## 12. Save Artifacts for Task 5

In [ ]:
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(selector,   'feature_selector.pkl')
joblib.dump(scaler,     'scaler.pkl')

print("Saved:")
print(f"  best_model.pkl       — {best_model_name}")
print("  feature_selector.pkl — SelectFromModel (RF-based)")
print("  scaler.pkl           — StandardScaler")
print("\nReady for Task 5: Model Evaluation & Interpretation.")


## Summary

| Stage | Action | Result |
|-------|--------|--------|
| Class imbalance | SMOTE (train only) | 4,139 samples per class |
| Scaling | StandardScaler | All 30 features normalised |
| Baseline (6 models) | CV F1 + test evaluation | SVM & GB led |
| Feature selection | RF importance, mean threshold | 7 features retained |
| Hyperparameter tuning | GridSearchCV, 5-fold CV, F1 | 3 models tuned |
| **Best model** | **Gradient Boosting (Tuned)** | **F1 = 0.5882, Recall = 0.6684** |

**Best params (Gradient Boosting):** `learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8`

The tuned Gradient Boosting model correctly identifies **~67% of churners** with a
precision of 0.53. In a churn-reduction context, recall is the priority metric —
missing a churner is more costly than a false positive. Artifacts saved for Task 5.
